# 11 — What survives (paper §8)

**Read-only report.** Net of the deconstruction, three things stand: (1) liquidations → future volatility (the consensus channel, confirmed and extended to the tails); (2) a **symmetric**, out-of-sample-validated short-horizon tail-risk indicator; (3) a **bounded** null — downside-specificity is excluded beyond the SESOI.

In [ ]:
# Setup — read-only report over canonical artefacts (data/econ, 2026-06-12)
import sys
from pathlib import Path
ROOT = Path.cwd().parent if (Path.cwd() / "..").resolve().name else Path.cwd()
ROOT = Path.cwd().parent  # notebooks/ -> repo root
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts" / "paper"))
sys.path.insert(0, str(ROOT / "scripts"))

import pandas as pd
pd.set_option("display.width", 140)
import style, make_figures as MF
style.apply()
from config import ECON_DIR
print("canonical dir:", ECON_DIR)

## 1. The volatility channel (A4)
Realised-volatility response positive and rising with horizon (rv: +0.021 at h=0 → +0.065 at h=12), aligned with OECD/Lehar-Parlour.

In [ ]:
vr = pd.read_csv(ECON_DIR / "vol_response.csv")
vr[vr.h.isin([0,1,3,6,12,24])].round(4)

## 2. The symmetric tail predictor (A8) — in-sample…
Both tails load positively (the exceedance LPM), Δ ≈ 0 everywhere (Fig 5 left).

In [ ]:
res = pd.read_csv(ECON_DIR / "exceedance_results.csv")
res[(res.method=="lpm") & (res.h.isin([0,1,12]))][["alpha","h","side","beta","ci_lo","ci_hi"]].round(5)

## 3. …and out-of-sample (Table 6)
Pinball skill vs the nested no-liquidation benchmark: positive in 20/20 cells, significant at both moderate tails at short horizons (τ=.05 h1 p=.004; τ=.95 h1 p<.001). Honest caveat: does **not** beat a dedicated GARCH(1,1) — the claim is *incremental information*, not dominance.

In [ ]:
oos = pd.read_csv(ECON_DIR / "oos_predictive.csv")
oos.assign(skill_pct=(100*oos.skill).round(2))[["benchmark","tau","h","skill_pct","dm_t","dm_pval"]].round({"dm_t":2,"dm_pval":3})

## 4. The bounded null (A5/A9) — Fig 7
Exceedance Δ@1% is equivalent-to-negligible under **all three** SESOI spans (strict included); the 1% skew whisper is statistically real (p=.012) but economically negligible under two spans and borderline under the strict one.

In [ ]:
mde = pd.read_csv(ECON_DIR / "mde_equivalence.csv")
mde

In [ ]:
MF.fig7_mde_equivalence()

## 5. Stability — splits, leave-out-Aug-2024, block length
The profile survives every split (~70% of the h=12 deepening without the Aug-2024 episode); Δ stays null per subsample; the MDE bound is insensitive to the MBB block length (12/24/36/48h; block=24 reproduces the headline exceedance cells exactly — built-in cross-check).

In [ ]:
sub = pd.read_csv(ECON_DIR / "subsample_stability.csv")
sub[sub.h.isin([0,12])].round(5)

In [ ]:
bl = pd.read_csv(ECON_DIR / "block_sensitivity.csv")
bl[bl.object=="delta_paired"][["block_size","h","estimate","ci_lo","ci_hi","mde80"]].round(5)